# M5.0a Primer: Current, Resistance and Conservation of Charge

Book pointer: chapter 9, sections 9.1 (current), 9.2 (Ohm's law on microscopic form), 9.3 (resistance), 9.4 (energy loss), 9.5 (conservation of charge).

First of four primers for module 5. Modules 2 to 4 dealt with charges at rest, or with steady currents taken as given. This primer asks what a current is, why a conductor carries a steady current when a field is applied instead of letting its charges accelerate forever, how the shape and material of a conductor set its resistance, where the electrical energy goes, and what conservation of charge says about how current can flow. Every result is derived by hand and then reproduced in code. The code uses `emlib.circuits.solve_sheet`, which finds the current in a flat conducting sheet of any shape. Notebook 01 of this module uses the same solver on harder geometries.

## 1. Warm-up

Try on paper first. The answers are in section 7.

1. A current of 1 A flows in a copper wire of cross-section 1 mm². Roughly how fast do the electrons move along the wire: micrometres per second, millimetres per second, metres per second, or kilometres per second?
2. What is the difference between the current $I$ and the current density $\mathbf{J}$? How do you get $I$ from $\mathbf{J}$?
3. A constant electric field acts on a free electron in vacuum and on a conduction electron in a metal. Why does the second reach a steady average velocity while the first keeps accelerating?
4. A wire is drawn out to twice its length at constant volume. By what factor does its resistance change?
5. Where in a resistor is electrical energy turned into heat, and at what rate per unit volume?
6. Write the equation that expresses conservation of charge at a point. What does it reduce to when the currents are steady?
7. Two wires of different metals are joined end to end and carry a steady current. Which is continuous across the junction, the current density or the electric field? What sits at the junction to make the other one jump?

## 2. The ideas

**Current is moving charge.** Until now the charges in this course have either sat still (electrostatics) or flowed in wires whose current was simply given (magnetostatics). A current is charge in motion, and the current $I$ through a surface is the charge that crosses it per unit time, measured in amperes (coulombs per second). It is a property of a surface, not of a point: the current through a wire is the same whichever cross-section you count at, but it tells you nothing about how the charge is distributed over that cross-section.

**Current density is the local quantity.** The current density $\mathbf{J}$ is a vector field. Its direction is the direction the positive charge flows (or opposite to the motion of electrons), and its magnitude is the current per unit area through a small surface perpendicular to it. The current through any surface is the flux of $\mathbf{J}$, $I = \int_S \mathbf{J}\cdot d\mathbf{S}$, which is the same flux integral that module 2 used for $\mathbf{E}$. If the carriers have density $n$, charge $q$ and average velocity $\mathbf{v}_d$, then $\mathbf{J} = nq\mathbf{v}_d$. In a metal $n$ is huge, so a large current needs only a tiny average velocity, the drift velocity, of a fraction of a millimetre per second.

**Why the current is steady.** A free electron in a constant field accelerates without limit. A conduction electron in a metal does not, because it keeps colliding with lattice vibrations and impurities. Between collisions the field adds a little velocity along $-\mathbf{E}$; each collision scrambles the direction and throws that gain away. The result is a steady average drift proportional to the field, on top of a fast random motion that averages to zero. The simplest version of this picture, the Drude model of section 3.2, predicts $\mathbf{J} = \sigma\mathbf{E}$ with a conductivity $\sigma$ set by the carrier density, charge, mass and the mean time between collisions.

**Ohm's law is a material law.** The relation $\mathbf{J} = \sigma\mathbf{E}$ is called Ohm's law on microscopic form. It is not a law of nature in the way Gauss' law is. It is a statement about how a particular material responds, like $\mathbf{P} = \epsilon_0\chi_e\mathbf{E}$ for a linear dielectric in module 3. Metals obey it extremely well over an enormous range of fields; diodes, gas discharges and semiconductors in strong fields do not. When a material obeys it, $\sigma$ is a single number (or a function of position, if the material varies) and the current flow problem becomes linear.

**Resistance is geometry plus material.** For a whole conductor between two contacts, the current is proportional to the voltage between the contacts, $V = RI$. The resistance $R$ depends on the material through $\sigma$ and on the shape. For a uniform wire it is $R = L/(\sigma A)$: longer wires resist more, thicker wires less. For other shapes it is found the way module 2 found capacitances: assume a current, find $\mathbf{J}$ from symmetry and conservation of charge, get $\mathbf{E} = \mathbf{J}/\sigma$, and integrate $\mathbf{E}$ along a path to find the voltage. When the shape has no symmetry, the computer does it, and that is what the sheet solver in this primer and notebook 01 are for.

**Energy goes to heat.** The field does work on the carriers as they drift, and the collisions hand that energy to the lattice as heat. The power delivered to a resistor is $P = IV = I^2R$, and locally the heating rate per unit volume is $\mathbf{J}\cdot\mathbf{E} = J^2/\sigma$. This is Joule heating: the reason a kettle boils, a fuse melts and power lines run at high voltage.

**Charge is conserved.** Charge cannot appear or disappear; it can only move. Applied to a small box this gives the continuity equation $\nabla\cdot\mathbf{J} + \partial\rho/\partial t = 0$: if current flows out of a box, the charge inside it must fall. In a steady state nothing changes in time, so $\nabla\cdot\mathbf{J} = 0$: as much current enters every region as leaves it. This is the continuous form of Kirchhoff's current law, and it is the single condition that makes circuits possible. Combined with $\mathbf{J} = \sigma\mathbf{E}$ and $\mathbf{E} = -\nabla V$, it gives $\nabla\cdot(\sigma\nabla V) = 0$, which is the same equation as the dielectric problem of module 3 with $\epsilon$ replaced by $\sigma$. Everything learned there about solving it on a grid carries over.

**What notebook 01 does.** It solves for the current in flat conducting sheets of non-uniform conductivity between two electrodes, measures their resistance, maps the current density and the Joule heating, and checks conservation of current at every node. The sheet is treated as a resistor network on a grid: each link between neighbouring nodes is a small resistor whose conductance is set by the local $\sigma$. That idea, a continuous conductor as a network of resistors, is chapter 10's "circuits as models" and returns in primer 00b and in the percolation notebook.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import circuits as ci
from emlib.fields import EPS0
from emlib.integrators import rk4_ode

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=4, suppress=True)
E_CHARGE, M_E = 1.602e-19, 9.109e-31          # C, kg
SIGMA_CU, N_CU = 5.96e7, 8.5e28                # S/m, conduction electrons per m^3

### 3.1 Current and current density (section 9.1)

**Current.** Pick a surface, for example a cross-section of a wire, and count the net charge $\Delta Q$ that crosses it in a time $\Delta t$, counting positive charge crossing in the chosen direction as positive and negative charge crossing in that direction as negative. The current through the surface is
$$I = \frac{dQ}{dt},$$
in amperes, $1\,\mathrm{A} = 1\,\mathrm{C/s}$. The sign convention matters once and then never again: current is defined as the direction positive charge would move. In a metal the carriers are electrons, which move the other way, but a stream of negative charge moving left is the same current as a stream of positive charge moving right.

**Current density.** The current through a surface says nothing about where on the surface the charge crosses. The local description is the current density $\mathbf{J}(\mathbf{r})$, a vector field. Take a tiny surface element $d\mathbf{S} = \hat{\mathbf{n}}\,dS$. The current through it is $\mathbf{J}\cdot d\mathbf{S}$: only the component of $\mathbf{J}$ along the normal counts, just as only the normal component of $\mathbf{E}$ counted for electric flux. For a finite surface,
$$I = \int_S \mathbf{J}\cdot d\mathbf{S}.$$
So the current is the flux of the current density, and everything primer M1.0b said about flux integrals applies. The units of $J$ are amperes per square metre.

**Current density from the carriers.** Suppose the carriers have number density $n$ (per cubic metre), charge $q$ each, and all move with the same average velocity $\mathbf{v}_d$. In a time $dt$ the carriers that cross a surface element $dS$ perpendicular to $\mathbf{v}_d$ are those in a small cylinder of base $dS$ and length $v_d\,dt$, so the charge crossing is $nq\,v_d\,dt\,dS$. Dividing by $dt\,dS$:
$$\mathbf{J} = nq\,\mathbf{v}_d.$$
Each carrier also has a fast random velocity, but random velocities point every way equally often and contribute nothing on average. Only the average velocity, the drift velocity, carries current. If several kinds of carrier are present (electrons and holes in a semiconductor, positive and negative ions in salt water), their contributions add.

**Worked example 1 (drift speed in copper).** A current of $1\,$A flows through a copper wire of cross-section $1\,\mathrm{mm^2} = 10^{-6}\,\mathrm{m^2}$, so $J = 10^6\,\mathrm{A/m^2}$. Copper has about one conduction electron per atom, $n = 8.5\times10^{28}\,\mathrm{m^{-3}}$. Then
$$v_d = \frac{J}{ne} = \frac{10^6}{8.5\times10^{28}\cdot1.602\times10^{-19}} = 7.3\times10^{-5}\,\mathrm{m/s},$$
about a tenth of a millimetre per second. An electron would need almost four hours to drift one metre. Compare two other speeds. The random thermal speed of a classical particle with the electron's mass at room temperature, $\sqrt{3kT/m}$, is $1.2\times10^5\,\mathrm{m/s}$; the real conduction electrons move even faster, about $1.6\times10^6\,\mathrm{m/s}$, for quantum reasons outside this course. And the signal that a switch has been closed travels along the wire at a large fraction of the speed of light, because it is carried by the field, which pushes on all the electrons in the wire at once. The drift is slow; the response is fast. Do not picture electrons racing from the battery to the bulb.

**Worked example 2 (non-uniform current density).** In a wire of radius $R$ the current density is not always uniform (at high frequency it crowds towards the surface, chapter 16 and beyond). Suppose $J(r) = J_0(1 - r^2/R^2)$ along the wire, largest on the axis and zero at the surface. The current is the flux through the cross-section, using rings of radius $r$ and width $dr$ with area $2\pi r\,dr$:
$$I = \int_0^R J_0\left(1 - \frac{r^2}{R^2}\right)2\pi r\,dr = 2\pi J_0\left(\frac{R^2}{2} - \frac{R^2}{4}\right) = \frac{\pi R^2 J_0}{2}.$$
With $J_0 = 2\times10^6\,\mathrm{A/m^2}$ and $R = 1\,$mm: $I = \pi\cdot10^{-6}\cdot10^6 = 3.14\,$A. The average current density is half the peak value. The code checks the ring sum numerically.

In [ ]:
J = 1.0 / 1e-6                                     # 1 A through 1 mm^2
v_d = J / (N_CU * E_CHARGE)
v_thermal = np.sqrt(3 * 1.381e-23 * 300 / M_E)
print(f"worked example 1: drift speed {v_d:.2e} m/s, time to drift 1 m {1 / v_d / 3600:.1f} h")
print(f"  classical thermal speed {v_thermal:.2e} m/s = {v_thermal / v_d:.1e} times the drift speed")

# Worked example 2: the current as a sum over rings.
J0, R = 2e6, 1e-3
r_edges = np.linspace(0, R, 2001)
r_mid = 0.5 * (r_edges[1:] + r_edges[:-1])
I_rings = np.sum(J0 * (1 - r_mid**2 / R**2) * 2 * np.pi * r_mid * np.diff(r_edges))
print(f"worked example 2: ring sum I = {I_rings:.5f} A,  by hand pi R^2 J0 / 2 = {np.pi * R**2 * J0 / 2:.5f} A")

**Animation.** Electrons in a short piece of wire jitter with fast random thermal motion, and the dashed cross-section counts the charge crossing it. With no field the crossings in the two directions cancel and the current readout stays at zero. When the field is switched on, the whole cloud acquires a slow drift against $\mathbf{E}$ and a net current appears. The drift is drawn exaggerated: the text in the scene gives the real ratio of thermal speed to drift speed, about a billion for copper at $1\,\mathrm{A/mm^2}$.

In [ ]:
from emlib import show
show.animation("m5_circuits", "DriftCurrent")

### 3.2 Ohm's law on microscopic form (section 9.2)

Why does a constant field produce a constant drift rather than a steadily growing one? The Drude model gives the answer with one assumption: each conduction electron moves freely between collisions, and a collision happens at random with probability $dt/\tau$ in any short time $dt$, where $\tau$ is the mean time between collisions. A collision resets the electron's velocity to a random thermal velocity with no preferred direction.

**The drift velocity.** Follow the average velocity $\langle\mathbf{v}\rangle$ of many electrons. In a short time $dt$ the field adds $(q\mathbf{E}/m)\,dt$ to every electron. A fraction $dt/\tau$ of them collide, and their velocities are replaced by random ones with zero average, so the average loses that fraction of itself. Together,
$$\frac{d\langle\mathbf{v}\rangle}{dt} = \frac{q\mathbf{E}}{m} - \frac{\langle\mathbf{v}\rangle}{\tau}.$$
This is the equation of motion of a particle with a drag force proportional to velocity. In the steady state the left side is zero and
$$\mathbf{v}_d = \frac{q\tau}{m}\mathbf{E} = \mu\mathbf{E},$$
where $\mu = |q|\tau/m$ is the mobility (velocity per unit field). The steady state is reached after a few $\tau$, which in a metal is around $10^{-14}\,$s, so for every practical purpose the response is instantaneous.

**The conductivity.** Substitute into $\mathbf{J} = nq\mathbf{v}_d$:
$$\mathbf{J} = \frac{nq^2\tau}{m}\mathbf{E} = \sigma\mathbf{E}, \qquad \sigma = \frac{nq^2\tau}{m}.$$
This is Ohm's law on microscopic form. The conductivity $\sigma$ is measured in siemens per metre ($\mathrm{S/m} = \mathrm{A/(V\,m)}$); its inverse $\rho = 1/\sigma$ is the resistivity in ohm metres. (The symbol $\rho$ is also used for charge density; context decides.) The charge enters squared, so the direction of $\mathbf{J}$ is along $\mathbf{E}$ whatever the sign of the carriers: negative electrons drift against the field, and their current flows with it.

**Worked example 3 (collision time in copper).** Solve $\sigma = ne^2\tau/m$ for $\tau$ with $\sigma = 5.96\times10^7\,\mathrm{S/m}$:
$$\tau = \frac{m\sigma}{ne^2} = \frac{9.109\times10^{-31}\cdot5.96\times10^7}{8.5\times10^{28}\cdot(1.602\times10^{-19})^2} = 2.5\times10^{-14}\,\mathrm{s}.$$
The mobility is $\mu = e\tau/m = 4.4\times10^{-3}\,\mathrm{m^2/(V\,s)}$. The field needed for $J = 10^6\,\mathrm{A/m^2}$ is $E = J/\sigma = 0.017\,\mathrm{V/m}$: tiny, which is why a copper wire looks like an equipotential in a circuit diagram. With the true electron speed of $1.6\times10^6\,\mathrm{m/s}$, the distance between collisions is about $40\,$nm, roughly 150 atomic spacings.

**Typical conductivities** at room temperature, in S/m:

| material | $\sigma$ | carriers |
|---|---|---|
| copper | $5.96\times10^7$ | electrons |
| aluminium | $3.8\times10^7$ | electrons |
| iron | $1.0\times10^7$ | electrons |
| graphite | about $10^5$ | electrons |
| seawater | about $5$ | ions |
| pure silicon | about $4\times10^{-4}$ | electrons and holes |
| glass | $10^{-11}$ to $10^{-15}$ | ions |

The range spans more than twenty orders of magnitude, wider than almost any other material property. That is why "conductor" and "insulator" are useful categories even though every material conducts a little.

**When Ohm's law fails.** The linear relation assumes $\tau$ and $n$ do not depend on the field. In a diode or transistor the carrier density depends exponentially on the voltage; in a gas, the field itself creates the carriers by ionisation; in very strong fields the carriers heat up and $\tau$ changes. None of these appear in this module, which treats only ohmic conductors.

The code computes the copper numbers and then runs the Drude model directly: many electrons, a field, random collisions with mean time $\tau$, random thermal velocities after each collision. The average velocity settles at $q\tau E/m$ as derived, to within the statistical noise of a finite sample, while the individual velocities stay ten times larger. The ratio is kept at ten so the drift is visible on the plot; in copper it is about $10^{10}$.

In [ ]:
tau = M_E * SIGMA_CU / (N_CU * E_CHARGE**2)
mu = E_CHARGE * tau / M_E
print(f"worked example 3: tau = {tau:.2e} s, mobility = {mu:.2e} m^2/(V s), "
      f"field for 1 A/mm^2 = {1e6 / SIGMA_CU:.4f} V/m, mean free path = {1.6e6 * tau * 1e9:.0f} nm")

# The Drude model, simulated in units where m = tau = |q| = E = 1. Electrons have q = -1.
rng = np.random.default_rng(0)
N, dt, steps, v_th = 100000, 0.01, 2500, 10.0
q, E_field = -1.0, 1.0
v = rng.normal(0, v_th, N)                          # x components of the velocity
mean_v = np.empty(steps)
for k in range(steps):
    v += q * E_field * dt                            # free acceleration
    hit = rng.random(N) < dt                         # collision probability dt / tau
    v[hit] = rng.normal(0, v_th, hit.sum())          # reset to a random thermal velocity
    mean_v[k] = v.mean()
t = dt * np.arange(1, steps + 1)
settled = mean_v[t > 5].mean()
print(f"Drude simulation: average velocity after t > 5 tau = {settled:.3f}   prediction q tau E / m = {q * E_field:.3f}")
print(f"  thermal spread {v_th}: the individual velocities are {v_th / abs(settled):.0f} times the drift")

plt.plot(t, mean_v, lw=0.8, label="average over 100000 electrons")
plt.plot(t, q * E_field * (1 - np.exp(-t)), "k--", label="q tau E / m (1 - exp(-t / tau))")
plt.xlabel("t / tau"); plt.ylabel("average velocity"); plt.legend(); plt.title("collisions turn acceleration into a steady drift"); plt.show()

### 3.3 Resistance (section 9.3)

**A uniform wire.** Take a wire of length $L$ and cross-section $A$, made of a material with conductivity $\sigma$, with a voltage $V$ between its ends. In the steady state the current density is uniform across the wire and along it (section 3.5 explains why), so $J = I/A$. Ohm's law gives the field $E = J/\sigma = I/(\sigma A)$, also uniform and along the wire. The voltage is the line integral of $E$ along the wire, $V = EL$. So
$$V = \frac{L}{\sigma A}I = RI, \qquad R = \frac{L}{\sigma A}.$$
The resistance in ohms ($\Omega = \mathrm{V/A}$) grows with length (the carriers must be pushed further) and falls with cross-section (more parallel paths). The macroscopic law $V = RI$ is what is usually called Ohm's law; it follows from the microscopic form for any ohmic conductor, with $R$ depending on the shape.

**Worked example 4 (a copper wire).** $L = 10\,$m, diameter $1\,$mm, so $A = \pi(0.5\times10^{-3})^2 = 7.85\times10^{-7}\,\mathrm{m^2}$:
$$R = \frac{10}{5.96\times10^7\cdot7.85\times10^{-7}} = 0.214\,\Omega.$$
A current of $10\,$A drops $2.1\,$V along it. Drawing the wire out to twice the length at constant volume halves $A$ and doubles $L$, multiplying $R$ by four.

**The general method.** For other shapes the recipe is the same one module 2 used to find capacitance, with $\mathbf{J}$ in place of $\mathbf{E}$:

1. Assume a current $I$ enters at one contact and leaves at the other.
2. Use symmetry and $\nabla\cdot\mathbf{J} = 0$ (current is neither created nor lost) to find $\mathbf{J}$ everywhere.
3. Get $\mathbf{E} = \mathbf{J}/\sigma$.
4. Integrate $\mathbf{E}$ from one contact to the other to find $V$, and divide: $R = V/I$.

**Worked example 5 (a coaxial shell).** Current flows radially from an inner cylinder of radius $a$ to an outer cylinder of radius $b$ through a material of conductivity $\sigma$, over a length $L$. This is the leakage path through the insulation of a coaxial cable. By cylindrical symmetry $\mathbf{J}$ is radial and, since the whole current $I$ crosses every cylinder of radius $r$ (area $2\pi rL$), $J = I/(2\pi rL)$. Then $E = I/(2\pi\sigma rL)$ and
$$V = \int_a^b \frac{I}{2\pi\sigma L}\frac{dr}{r} = \frac{I}{2\pi\sigma L}\ln\frac{b}{a}, \qquad R = \frac{\ln(b/a)}{2\pi\sigma L}.$$
For polyethylene insulation ($\sigma \approx 10^{-15}\,\mathrm{S/m}$) with $a = 0.5\,$mm, $b = 2\,$mm and $L = 100\,$m: $R = \ln4/(2\pi\cdot10^{-15}\cdot100) = 2.2\times10^{12}\,\Omega$. The logarithm is the same one that appeared in the coaxial capacitance of primer M2.0c, and section 3.3's last paragraph explains why.

**Worked example 6 (a ground electrode).** A metal hemisphere of radius $a$ is buried flat side up at the surface of soil of conductivity $\sigma$, and a current $I$ flows from it into the ground and away to a distant return point. The current spreads over hemispheres of radius $r$ with area $2\pi r^2$, so $J = I/(2\pi r^2)$ and $E = I/(2\pi\sigma r^2)$. The potential relative to far away is
$$V(r) = \int_r^\infty \frac{I}{2\pi\sigma r'^2}dr' = \frac{I}{2\pi\sigma r}, \qquad R = \frac{V(a)}{I} = \frac{1}{2\pi\sigma a}.$$
With $\sigma = 0.01\,\mathrm{S/m}$ (moist soil) and $a = 0.5\,$m: $R = 31.8\,\Omega$. Most of the resistance is close to the electrode, where the current is crowded into a small area. This has a safety consequence: if a fault current of $1000\,$A flows into the electrode, the ground $5\,$m away sits at $1000/(2\pi\cdot0.01\cdot5) = 3183\,$V and the ground $5.8\,$m away at $2744\,$V. A person standing with feet $0.8\,$m apart across that gradient has $439\,$V between them, the "step voltage" that kills cattle near lightning strikes.

**Layers in series and in parallel.** A conductor made of slabs of different material stacked along the current is a series combination: the same current crosses every slab, the voltages add, and $R = \sum_k L_k/(\sigma_k A)$. Slabs side by side, each spanning the full length, are a parallel combination: the same voltage acts on each, the currents add, and $1/R = \sum_k \sigma_k A_k/L$. These are the familiar series and parallel rules, derived here from $\mathbf{J} = \sigma\mathbf{E}$ rather than assumed. When the conductivity varies smoothly, the sums become integrals, $R = \int dx/(\sigma(x)A)$ for variation along the current.

**Resistance and capacitance.** For any pair of electrodes embedded in a uniform medium of permittivity $\epsilon$ and conductivity $\sigma$, the field pattern is the same whether the medium is treated as a dielectric or as a conductor, because both $\mathbf{D} = \epsilon\mathbf{E}$ and $\mathbf{J} = \sigma\mathbf{E}$ are proportional to the same $\mathbf{E}$. The capacitance is $C = Q/V$ with $Q = \oint\mathbf{D}\cdot d\mathbf{S} = \epsilon\oint\mathbf{E}\cdot d\mathbf{S}$ over a surface around one electrode, and the resistance is $R = V/I$ with $I = \oint\mathbf{J}\cdot d\mathbf{S} = \sigma\oint\mathbf{E}\cdot d\mathbf{S}$ over the same surface. Divide:
$$RC = \frac{\epsilon}{\sigma}.$$

**Worked example 7 (checking $RC = \epsilon/\sigma$).** Parallel plates: $C = \epsilon A/d$ and $R = d/(\sigma A)$, product $\epsilon/\sigma$. Coaxial cylinders: $C = 2\pi\epsilon L/\ln(b/a)$ and, from worked example 5, $R = \ln(b/a)/(2\pi\sigma L)$, product $\epsilon/\sigma$ again. Numbers: a capacitor with a glass dielectric ($\epsilon_r = 5$, $\sigma = 10^{-12}\,\mathrm{S/m}$) loses its charge through the glass with time constant $RC = 5\cdot8.854\times10^{-12}/10^{-12} = 44\,$s, whatever its shape. The same product will reappear as the charge relaxation time in section 3.5.

The code evaluates worked examples 4 to 7 and checks the two radial resistances by adding up thin shells: $dR = dr/(\sigma\cdot\text{area})$ in series from the inner to the outer radius.

In [ ]:
R_wire = 10 / (SIGMA_CU * np.pi * 0.5e-3**2)
print(f"worked example 4: copper wire R = {R_wire:.4f} ohm, drop at 10 A = {10 * R_wire:.2f} V")

def shells(area, r_in, r_out, sigma, n=200000):
    '''Series sum of thin shells dR = dr / (sigma area(r)), geometric spacing to resolve small r.'''
    r = np.geomspace(r_in, r_out, n + 1); rm = np.sqrt(r[1:] * r[:-1])
    return np.sum(np.diff(r) / (sigma * area(rm)))

a, b, sig_pe, L = 0.5e-3, 2e-3, 1e-15, 100.0
print(f"worked example 5: coax shell by hand {np.log(b / a) / (2 * np.pi * sig_pe * L):.3e} ohm, "
      f"shell sum {shells(lambda r: 2 * np.pi * r * L, a, b, sig_pe):.3e} ohm")

sig_soil, a_e = 0.01, 0.5
print(f"worked example 6: ground electrode by hand {1 / (2 * np.pi * sig_soil * a_e):.2f} ohm, "
      f"shell sum to 10 km {shells(lambda r: 2 * np.pi * r**2, a_e, 1e4, sig_soil):.2f} ohm")
V_at = lambda r: 1000 / (2 * np.pi * sig_soil * r)
print(f"  1000 A fault: V(5 m) = {V_at(5):.0f} V, V(5.8 m) = {V_at(5.8):.0f} V, step voltage {V_at(5) - V_at(5.8):.0f} V")
print(f"worked example 7: RC for glass = {5 * EPS0 / 1e-12:.1f} s")

**A sheet as a resistor network.** When the conductor has no symmetry, the four-step recipe still works but step 2 needs a computer. `emlib.circuits.solve_sheet` handles a flat sheet of thickness $t$ lying in the $xy$ plane, with conductivity $\sigma(x, y)$ given on a grid of $n_x\times n_y$ nodes with spacing $h$. The left column of nodes is an electrode at $V_{\text{left}}$, the right column an electrode at $V_{\text{right}}$, and the top and bottom edges are insulating: no current crosses them.

The method treats the sheet as a resistor network. Each pair of neighbouring nodes is joined by a resistor representing the strip of material between them: length $h$, cross-section $h\times t$, so conductance $G = \sigma_{\text{face}}\,ht/h = \sigma_{\text{face}}t$. The spacing cancels, which is why only $\sigma$ and $t$ appear. The conductivity on a face is the harmonic mean of its two nodes, $2\sigma_1\sigma_2/(\sigma_1 + \sigma_2)$, which is exactly two half-cells of the two materials in series; it is the same choice module 3 made for permittivity, and for the same reason (it keeps the normal current continuous across an interface). Setting $\sigma = 0$ at a node makes every face touching it non-conducting, which is how holes are cut. Kirchhoff's current law, the discrete form of $\nabla\cdot\mathbf{J} = 0$, is then imposed at every free node, and the resulting sparse linear system is solved directly, exactly as in primer M2.0b.

One bookkeeping point. With insulating top and bottom edges, each of the $n_y$ rows of nodes carries its own channel of width $h$, so the sheet's effective width is $W = n_yh$ and its length between the electrodes is $L = (n_x - 1)h$. A uniform sheet then has $R = L/(\sigma Wt) = (n_x - 1)/(\sigma n_y t)$ exactly.

The code checks three cases against the hand formulas: a uniform strip, two layers in series along the current, and two layers side by side.

In [ ]:
nx, ny = 41, 11
V, Ix, Iy, I = ci.solve_sheet(np.ones((nx, ny)))
print(f"uniform strip: R = {1 / I:.4f}   formula (nx - 1) / (sigma ny t) = {(nx - 1) / ny:.4f}   "
      f"V linear in x: {np.allclose(V, np.linspace(1, 0, nx)[:, None])}")

# Series: sigma = 1 for nodes 0..20, 4 for nodes 21..40. The interface sits halfway between nodes 20 and 21.
sig = np.ones((nx, ny)); sig[21:, :] = 4.0
_, Ix, _, I = ci.solve_sheet(sig)
R_series = (20.5 / 1.0 + 19.5 / 4.0) / ny
print(f"series layers: R = {1 / I:.5f}   sum of L_k / (sigma_k W t) = {R_series:.5f}   "
      f"same current through every column of faces: {np.allclose(Ix.sum(axis=1), I)}")

# Parallel: the top half of the rows has sigma = 3, the bottom half sigma = 1.
sig = np.ones((nx, ny)); sig[:, 6:] = 3.0
_, _, Iy, I = ci.solve_sheet(sig)
G_parallel = (6 * 1.0 + 5 * 3.0) / (nx - 1)
print(f"parallel layers: 1/R = {I:.5f}   sum of sigma_k W_k t / L = {G_parallel:.5f}   "
      f"no current between the layers: {np.abs(Iy).max():.1e}")

### 3.4 Energy loss (section 9.4)

**Power delivered to a conductor.** A charge $dq$ that moves through a potential drop $V$ loses potential energy $V\,dq$. In a steady current, a charge $dq = I\,dt$ passes through the conductor in each time $dt$, so the electrical energy delivered per unit time is
$$P = IV.$$
In a resistor that energy does not accelerate the carriers (their drift speed is constant); the collisions hand it to the lattice as heat. With $V = RI$,
$$P = I^2R = \frac{V^2}{R}.$$
The two forms suggest opposite trends, and both are right: at fixed current, more resistance means more heat; at fixed voltage, more resistance means less current and less heat.

**Locally: Joule heating.** Take a small cylinder of material, length $dl$ along $\mathbf{J}$ and cross-section $dA$. The current through it is $J\,dA$ and the voltage across it is $E\,dl$, so the power is $JE\,dA\,dl$. Per unit volume,
$$p = \mathbf{J}\cdot\mathbf{E} = \sigma E^2 = \frac{J^2}{\sigma}.$$
This is the local Joule heating rate, in watts per cubic metre. Where the current crowds (high $J$), the heating is strongest, and it grows with the square of the crowding.

**Worked example 8 (a kettle).** A $1000\,$W kettle on $230\,$V mains draws $I = P/V = 4.35\,$A and has an element resistance $R = V^2/P = 52.9\,\Omega$. Heating one litre of water from $20\,^\circ$C to $100\,^\circ$C takes $4186\cdot1\cdot80 = 3.35\times10^5\,$J, about $335\,$s if no heat is lost.

**Worked example 9 (why power lines run at high voltage).** A town needs $P = 10\,$MW, delivered through lines with total resistance $5\,\Omega$. At $10\,$kV the line current is $I = P/V = 1000\,$A and the loss in the lines is $I^2R = 5\,$MW, half the power. At $400\,$kV the current is $25\,$A and the loss is $25^2\cdot5 = 3.1\,$kW, three hundredths of a percent. The delivered power is fixed, so the current falls as $1/V$ and the loss as $1/V^2$. Transformers, which need alternating current and chapter 14's induction, make the voltage change possible.

**Worked example 10 (the local and global forms agree).** A copper wire of cross-section $1\,\mathrm{mm^2}$ carries $10\,$A, so $J = 10^7\,\mathrm{A/m^2}$ and $p = J^2/\sigma = 10^{14}/5.96\times10^7 = 1.68\times10^6\,\mathrm{W/m^3}$. One metre of wire has volume $10^{-6}\,\mathrm{m^3}$, so it dissipates $1.68\,$W. From the global form: $R = 1/(5.96\times10^7\cdot10^{-6}) = 0.0168\,\Omega$ and $I^2R = 1.68\,$W. In general, for a uniform wire, $\int p\,dv = (I/A)^2/\sigma\cdot AL = I^2L/(\sigma A) = I^2R$.

In [ ]:
P, Vm = 1000.0, 230.0
print(f"worked example 8: I = {P / Vm:.2f} A, R = {Vm**2 / P:.1f} ohm, time to heat 1 L by 80 K = {4186 * 80 / P:.0f} s")
for V_line in (10e3, 400e3):
    I_line = 10e6 / V_line
    print(f"worked example 9: at {V_line / 1e3:5.0f} kV, I = {I_line:7.1f} A, line loss = {I_line**2 * 5 / 1e3:9.1f} kW "
          f"({100 * I_line**2 * 5 / 10e6:.3f} percent)")
J10 = 10 / 1e-6
print(f"worked example 10: p = {J10**2 / SIGMA_CU:.3e} W/m^3, per metre {J10**2 / SIGMA_CU * 1e-6:.3f} W, "
      f"I^2 R = {10**2 / (SIGMA_CU * 1e-6):.3f} W")

**Current crowding at a constriction.** The sheet solver makes the local picture visible. Take a uniform sheet and cut two slots in from the top and bottom edges, leaving a narrow neck in the middle. All the current must pass through the neck, so $J$ is high there, and since $p = J^2/\sigma$ the heating is concentrated even more strongly. This is how a fuse works: a deliberate constriction where the heating is greatest. It is also why the resistance of a constricted sheet is much larger than the fraction of material removed would suggest.

On the grid, the current through each face is $I_{\text{face}} = G_{\text{face}}(V_i - V_{i+1})$ and its Joule power is $I_{\text{face}}^2/G_{\text{face}}$. The sum of the face powers must equal $IV$ for the whole sheet, which is energy conservation and a check on the solution. The code also computes the net current out of every free node, which Kirchhoff's law says must vanish, and maps $|\mathbf{J}|$ and $p$ (each face's value is shared between its two nodes, averaged per node).

In [ ]:
def face_conductances(sigma, t=1.0):
    '''Harmonic-mean face conductances as used by solve_sheet: Gx (nx-1, ny), Gy (nx, ny-1).'''
    def hm(s1, s2):
        with np.errstate(divide="ignore", invalid="ignore"):
            return np.where((s1 > 0) & (s2 > 0), 2 * s1 * s2 / (s1 + s2), 0.0)
    return hm(sigma[:-1], sigma[1:]) * t, hm(sigma[:, :-1], sigma[:, 1:]) * t

def node_net_current(Ix, Iy):
    nx, ny = Ix.shape[0] + 1, Iy.shape[1] + 1
    net = np.zeros((nx, ny))
    net[:-1, :] += Ix; net[1:, :] -= Ix; net[:, :-1] += Iy; net[:, 1:] -= Iy
    return net

def to_nodes(fx, fy):
    '''Average face quantities onto the nodes they touch (for maps).'''
    nx, ny = fx.shape[0] + 1, fy.shape[1] + 1
    s, c = np.zeros((nx, ny)), np.zeros((nx, ny))
    s[:-1] += fx; s[1:] += fx; c[:-1] += 1; c[1:] += 1
    s[:, :-1] += fy; s[:, 1:] += fy; c[:, :-1] += 1; c[:, 1:] += 1
    return s / c

nx, ny = 81, 41
sig = np.ones((nx, ny))
sig[38:43, :15] = 0.0; sig[38:43, 26:] = 0.0                 # two slots leave a neck 11 rows wide
V, Ix, Iy, I = ci.solve_sheet(sig)
Gx, Gy = face_conductances(sig)
with np.errstate(divide="ignore", invalid="ignore"):
    px, py = np.where(Gx > 0, Ix**2 / Gx, 0.0), np.where(Gy > 0, Iy**2 / Gy, 0.0)
P_faces = px.sum() + py.sum()
net = node_net_current(Ix, Iy)
print(f"resistance with the neck: {1 / I:.3f}   without: {(nx - 1) / ny:.3f}   "
      f"(material removed: {100 * (sig == 0).mean():.1f} percent)")
print(f"sum of face powers = {P_faces:.6f}   I V = {I:.6f}")
print(f"largest net current out of a free node: {np.abs(net[1:-1]).max():.1e}  (Kirchhoff: zero)")

J_nodes = to_nodes(np.abs(Ix), np.abs(Iy))                 # h = t = 1, so face current = current density
p_nodes = to_nodes(px, py)
x = np.arange(nx); y = np.arange(ny)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
im = axes[0].contourf(x, y, V.T, levels=21, cmap="viridis"); fig.colorbar(im, ax=axes[0], label="V")
axes[0].streamplot(x, y, to_nodes(Ix, 0 * Iy).T, to_nodes(0 * Ix, Iy).T, color="w", density=1.0, linewidth=0.6)
axes[0].set_title("potential and current lines")
im = axes[1].imshow(np.ma.masked_where(sig == 0, J_nodes).T, origin="lower", cmap="magma"); fig.colorbar(im, ax=axes[1], label="|J|")
axes[1].set_title("current density crowds into the neck")
im = axes[2].imshow(np.ma.masked_where(sig == 0, p_nodes).T, origin="lower", cmap="inferno"); fig.colorbar(im, ax=axes[2], label="p = J^2 / sigma")
axes[2].set_title("Joule heating: square of the crowding")
for ax in axes: ax.set_aspect("equal"); ax.set_xlabel("x / h"); ax.set_ylabel("y / h")
plt.tight_layout(); plt.show()
print(f"peak |J| in the neck / uniform |J| far away: {J_nodes[40, 20] / J_nodes[5, 20]:.2f};  "
      f"heating ratio {p_nodes[40, 20] / p_nodes[5, 20]:.2f}")

### 3.5 Conservation of charge (section 9.5)

**The continuity equation.** Charge is conserved: the total charge in a closed region changes only by charge crossing its boundary. Take any fixed volume $v$ with closed surface $S$. The current flowing out through the surface is $\oint_S\mathbf{J}\cdot d\mathbf{S}$, and it must equal the rate at which the charge inside falls:
$$\oint_S \mathbf{J}\cdot d\mathbf{S} = -\frac{d}{dt}\int_v \rho\,dv.$$
This is the integral form. Apply the divergence theorem (primer M1.0d) to the left side, move everything under one integral, and use the fact that the region was arbitrary:
$$\nabla\cdot\mathbf{J} + \frac{\partial\rho}{\partial t} = 0.$$
This is the continuity equation, conservation of charge at a point. Read it like the divergence in primer M1.0c: where current spreads out of a point ($\nabla\cdot\mathbf{J} > 0$), the charge there is being drained.

**Steady currents.** In a steady state nothing depends on time, so
$$\nabla\cdot\mathbf{J} = 0, \qquad \oint_S\mathbf{J}\cdot d\mathbf{S} = 0 \text{ for every closed surface.}$$
Every region receives exactly as much current as it gives away. Applied to a small box around a junction of wires, it says the currents flowing into the junction sum to zero: Kirchhoff's current law, the starting point of primer 00b. Applied to a wire, it says the current is the same through every cross-section, which is what worked example 4 assumed. It is also what makes Ampère's law in module 4 consistent: the circulation of $\mathbf{B}$ counted the current through any surface bounded by the loop, and only a divergence-free current gives the same answer for every surface.

**The steady-state equation.** Combine $\nabla\cdot\mathbf{J} = 0$ with $\mathbf{J} = \sigma\mathbf{E}$ and $\mathbf{E} = -\nabla V$:
$$\nabla\cdot(\sigma\nabla V) = 0.$$
Where $\sigma$ is uniform this is Laplace's equation, so everything in primer M2.0b about uniqueness and relaxation applies to current flow. Where $\sigma$ varies, it is the equation of module 3 with $\epsilon$ replaced by $\sigma$. The sheet solver is the discrete form of this equation.

**Boundary conditions.** At an interface between two conductors, apply the steady integral law to a thin pillbox straddling the interface: the flux in through one face equals the flux out through the other, so the normal component of $\mathbf{J}$ is continuous, $J_{1n} = J_{2n}$. The tangential component of $\mathbf{E}$ is continuous as in electrostatics (curl-free field, small loop argument), so $J_{1t}/\sigma_1 = J_{2t}/\sigma_2$. At an insulating boundary $\sigma_2 = 0$, so $J_n = 0$: current runs parallel to the surface of a conductor surrounded by insulator, which is the condition `solve_sheet` imposes on its top and bottom edges.

**Surface charge at an interface.** If $J_n$ is continuous but $\sigma$ jumps, then $E_n = J_n/\sigma$ jumps. By Gauss' law, a jump in the normal field requires a surface charge, $\sigma_s = \epsilon_0(E_{2n} - E_{1n})$ (for conductors with $\epsilon \approx \epsilon_0$). So whenever current crosses from a good conductor into a poorer one, a thin layer of charge sits at the junction. This charge is not a side effect; it is what creates the stronger field in the poorer conductor that drives the same current through it. More generally, surface charges on the wires of a circuit are what steer the field along the wires and around corners.

**Worked example 11 (a copper to iron junction).** $J = 10^6\,\mathrm{A/m^2}$ flows from copper ($\sigma_1 = 5.96\times10^7$) into iron ($\sigma_2 = 10^7$). The fields are $E_1 = 0.0168\,$V/m and $E_2 = 0.1\,$V/m, and the surface charge is
$$\sigma_s = \epsilon_0 J\left(\frac{1}{\sigma_2} - \frac{1}{\sigma_1}\right) = 8.854\times10^{-12}\cdot10^6\cdot(10^{-7} - 1.68\times10^{-8}) = 7.4\times10^{-13}\,\mathrm{C/m^2},$$
about five electrons per square millimetre: an almost invisible amount of charge doing an essential job.

**Charge relaxation.** What happens to a lump of excess charge placed inside a conductor? Combine continuity with Ohm's law and Gauss' law: $\nabla\cdot\mathbf{J} = \sigma\nabla\cdot\mathbf{E} = \sigma\rho/\epsilon$, so
$$\frac{\partial\rho}{\partial t} = -\frac{\sigma}{\epsilon}\rho, \qquad \rho(t) = \rho_0e^{-t/\tau_r}, \qquad \tau_r = \frac{\epsilon}{\sigma}.$$
The charge density decays exponentially everywhere inside, and the charge reappears on the surface, which is the electrostatic result of module 2 (no charge inside a conductor) with a time scale attached. The relaxation time is the same $\epsilon/\sigma$ as the $RC$ product of worked example 7.

**Worked example 12 (relaxation times).** Copper, with $\epsilon \approx \epsilon_0$: $\tau_r = 8.854\times10^{-12}/5.96\times10^7 = 1.5\times10^{-19}\,$s. (This is shorter than the collision time $\tau = 2.5\times10^{-14}\,$s, so the Drude model's assumption of instantaneous response fails here and the true relaxation is set by plasma oscillations; the conclusion, that charge leaves the interior of a metal essentially at once, stands.) Seawater, $\epsilon_r \approx 80$, $\sigma \approx 4.8\,$S/m: $\tau_r = 80\cdot8.854\times10^{-12}/4.8 = 1.5\times10^{-10}\,$s. Glass from worked example 7: $44\,$s. Whether a material behaves as a conductor or an insulator depends on comparing $\tau_r$ with the time scale of the problem: seawater is a conductor for radio waves below about a gigahertz and an insulator above it.

The code integrates the relaxation equation with `rk4_ode` and compares it with the exponential, then measures the junction charge of worked example 11 on a sheet: two halves of copper and iron, the field on each side from the potential differences, and $\epsilon_0$ times the jump.

In [ ]:
# Charge relaxation in seawater.
eps_sea, sig_sea = 80 * EPS0, 4.8
tau_r = eps_sea / sig_sea
t = np.linspace(0, 5 * tau_r, 201)
rho = rk4_ode(lambda tk, r: -sig_sea / eps_sea * r, 1.0, t)
print(f"worked example 12: tau_r(seawater) = {tau_r:.2e} s, tau_r(copper) = {EPS0 / SIGMA_CU:.2e} s; "
      f"rk4 vs exp(-t / tau_r): max difference {np.abs(rho - np.exp(-t / tau_r)).max():.1e}")

# Worked example 11 on a sheet: copper for nodes 0..20, iron for nodes 21..40, h = 1 mm, t = 1 mm.
nx, ny, h, thick = 41, 5, 1e-3, 1e-3
sig = np.full((nx, ny), SIGMA_CU); sig[21:, :] = 1.0e7
V_drop = 1e6 * h * (20.5 / SIGMA_CU + 19.5 / 1.0e7)   # the voltage that drives J = 1e6 A/m^2 through both metals
V, Ix, Iy, I = ci.solve_sheet(sig, V_left=V_drop, V_right=0.0, thickness=thick)
J_face = Ix[:, 2] / (h * thick)                     # current density through each x face of the middle row
E_face = (V[:-1, 2] - V[1:, 2]) / h                 # field on each x face
E_cu, E_fe = E_face[10], E_face[30]                 # well inside each metal
sigma_s = EPS0 * (E_fe - E_cu)
print(f"worked example 11 on a sheet: J = {J_face[10]:.4e} A/m^2 in copper and {J_face[30]:.4e} in iron (continuous)")
print(f"  E_cu = {E_cu:.4e} V/m, E_fe = {E_fe:.4e} V/m, ratio {E_fe / E_cu:.3f} = sigma_cu / sigma_fe = {SIGMA_CU / 1e7:.3f}")
print(f"  junction charge eps0 (E_fe - E_cu) = {sigma_s:.3e} C/m^2   formula eps0 J (1/sigma_fe - 1/sigma_cu) = "
      f"{EPS0 * J_face[10] * (1 / 1e7 - 1 / SIGMA_CU):.3e}")

### Exercise: the resistance of a strip

Replace the body of `strip_resistance` with your own expression for the resistance of a uniform rectangular strip of length $L$ (along the current), width $W$, thickness $t$ and conductivity $\sigma$. The check builds the same strip on a grid, remembering that the effective width of a grid of $n_y$ rows is $n_yh$ and its length is $(n_x - 1)h$, and compares with `solve_sheet` for two strips.

In [ ]:
def strip_resistance(L, W, sigma, t):
    '''Resistance of a uniform strip: length L along the current, width W, thickness t, conductivity sigma.'''
    return L / (sigma * W * t)                                  # replace with your own version


for nx, ny, h, sigma, t in ((51, 21, 2e-3, 3.0, 1e-3), (31, 61, 5e-4, 5.96e7, 35e-6)):
    _, _, _, I = ci.solve_sheet(np.full((nx, ny), sigma), thickness=t)
    R_grid, R_you = 1 / I, strip_resistance((nx - 1) * h, ny * h, sigma, t)
    print(f"strip {(nx - 1) * h * 1e3:.1f} mm x {ny * h * 1e3:.1f} mm, sigma {sigma:.3g} S/m, t {t * 1e6:.0f} um: "
          f"grid R = {R_grid:.6g} ohm, yours = {R_you:.6g} ohm")
    assert np.isclose(R_grid, R_you, rtol=1e-10)

## 4. Verify

Two checks with asserts.

1. Series layers. The resistance of a sheet made of three layers along the current must equal $\sum_k L_k/(\sigma_kWt)$, with each layer's length measured to the point halfway between the last node of one material and the first node of the next. That is where the harmonic-mean face puts the interface: half a cell of each material in series.
2. Energy conservation. For the constricted sheet of section 3.4 with a non-uniform conductivity added, the Joule power summed over every face must equal $IV$ for the whole sheet, and the net current out of every free node must vanish.

In [ ]:
# 1. Three layers: sigma = 1 (nodes 0..15), 5 (16..30), 2 (31..50).
nx, ny = 51, 9
sig = np.ones((nx, ny)); sig[16:31] = 5.0; sig[31:] = 2.0
_, _, _, I = ci.solve_sheet(sig)
lengths = np.array([15.5, 15.0, 19.5])               # node 0 to 15.5, 15.5 to 30.5, 30.5 to 50
R_hand = np.sum(lengths / np.array([1.0, 5.0, 2.0])) / ny
print(f"three layers: R = {1 / I:.6f}   sum L_k / (sigma_k W t) = {R_hand:.6f}")
assert np.isclose(1 / I, R_hand, rtol=1e-12)

# 2. Constriction plus a smooth conductivity gradient.
nx, ny = 81, 41
xx, yy = np.meshgrid(np.arange(nx), np.arange(ny), indexing="ij")
sig = 1.0 + 2.0 * np.exp(-((xx - 60) ** 2 + (yy - 10) ** 2) / 80.0)
sig[38:43, :15] = 0.0; sig[38:43, 26:] = 0.0
V, Ix, Iy, I = ci.solve_sheet(sig, V_left=2.0, V_right=0.5)
Gx, Gy = face_conductances(sig)
with np.errstate(divide="ignore", invalid="ignore"):
    P_faces = np.sum(np.where(Gx > 0, Ix**2 / Gx, 0)) + np.sum(np.where(Gy > 0, Iy**2 / Gy, 0))
print(f"Joule power over all faces = {P_faces:.10f}   I (V_left - V_right) = {I * 1.5:.10f}")
print(f"largest net current out of a free node = {np.abs(node_net_current(Ix, Iy)[1:-1]).max():.1e}")
assert np.isclose(P_faces, I * 1.5, rtol=1e-10)
assert np.abs(node_net_current(Ix, Iy)[1:-1]).max() < 1e-12

## 5. Explore and rebuild

Explore. Cut a single circular hole of radius $a$ in the middle of a wide sheet and measure how much the resistance rises as $a$ grows, compared with the fraction of area removed. Then replace the hole by an inclusion of very high conductivity (say $\sigma = 1000$) of the same size. The insulating hole forces current around it and the conducting inclusion draws current into it; the resistance changes in the two cases are not equal and opposite. Map $|\mathbf{J}|$ and the Joule heating for both, and look at where the current crowds: at the sides of the hole, at the front and back of the inclusion. For a small circular inclusion in a wide sheet the theory of module 3 (the dielectric cylinder, with $\epsilon$ replaced by $\sigma$) predicts the field inside; check it.

Rebuild. Close this notebook. On paper: derive $\mathbf{J} = nq\mathbf{v}_d$ from counting carriers in a cylinder, derive $\sigma = nq^2\tau/m$ from the Drude equation of motion, derive $R = L/(\sigma A)$ and the coaxial and hemispherical resistances, derive $p = \mathbf{J}\cdot\mathbf{E}$, and derive the continuity equation from a box and the relaxation time $\epsilon/\sigma$. In a blank editor, write the Drude simulation and a function that turns a conductivity grid into face conductances, then reproduce worked example 11 on a sheet.

## 6. Key results

| result | formula | holds when |
|---|---|---|
| current | $I = dQ/dt = \int_S\mathbf{J}\cdot d\mathbf{S}$ | any surface |
| current density from carriers | $\mathbf{J} = nq\mathbf{v}_d$, summed over carrier types | $\mathbf{v}_d$ the average velocity |
| drift speed, copper | $7.3\times10^{-5}\,$m/s at $1\,\mathrm{A/mm^2}$ | $n = 8.5\times10^{28}\,\mathrm{m^{-3}}$ |
| Drude equation | $d\langle\mathbf{v}\rangle/dt = q\mathbf{E}/m - \langle\mathbf{v}\rangle/\tau$ | random collisions, mean time $\tau$ |
| drift and mobility | $\mathbf{v}_d = (q\tau/m)\mathbf{E}$, $\mu = |q|\tau/m$ | steady state, after a few $\tau$ |
| Ohm's law, microscopic | $\mathbf{J} = \sigma\mathbf{E}$, $\sigma = nq^2\tau/m$, resistivity $1/\sigma$ | ohmic material |
| uniform wire | $R = L/(\sigma A)$ | uniform $\sigma$ and cross-section |
| coaxial shell, radial | $R = \ln(b/a)/(2\pi\sigma L)$ | current from radius $a$ to $b$ |
| hemispherical electrode | $R = 1/(2\pi\sigma a)$, $V(r) = I/(2\pi\sigma r)$ | half-space of conductivity $\sigma$ |
| series and parallel layers | $R = \sum L_k/(\sigma_kA)$, $1/R = \sum\sigma_kA_k/L$ | layers across or along the current |
| resistance and capacitance | $RC = \epsilon/\sigma$ | same electrodes, uniform medium |
| power | $P = IV = I^2R = V^2/R$ | any two-terminal conductor |
| Joule heating density | $p = \mathbf{J}\cdot\mathbf{E} = J^2/\sigma$ | ohmic material |
| continuity equation | $\nabla\cdot\mathbf{J} + \partial\rho/\partial t = 0$ | always (charge conservation) |
| steady currents | $\nabla\cdot\mathbf{J} = 0$, $\nabla\cdot(\sigma\nabla V) = 0$ | time-independent |
| interface conditions | $J_n$ continuous, $E_t$ continuous, $J_n = 0$ at an insulator | steady state |
| junction charge | $\sigma_s = \epsilon_0J_n(1/\sigma_2 - 1/\sigma_1)$ | current from medium 1 into 2 |
| charge relaxation | $\rho = \rho_0e^{-t/\tau_r}$, $\tau_r = \epsilon/\sigma$ | uniform ohmic medium |
| sheet on a grid | face conductance $\sigma_{\text{face}}t$, harmonic mean; $W = n_yh$, $L = (n_x - 1)h$ | `solve_sheet` |

## 7. Warm-up answers

1. Millimetres per second or slower: worked example 1 gives $7\times10^{-5}\,$m/s, a tenth of a millimetre per second. The signal travels near the speed of light because the field acts on all the electrons at once.
2. $I$ is the charge per unit time through a whole surface, a single number; $\mathbf{J}$ is a vector field giving the current per unit area at each point. $I = \int_S\mathbf{J}\cdot d\mathbf{S}$.
3. Collisions with the lattice and impurities scramble the velocity every $\tau \approx 10^{-14}\,$s, throwing away the velocity gained from the field. The balance between the field's push and this loss gives a steady drift $q\tau\mathbf{E}/m$.
4. By a factor of four: $L$ doubles and, at constant volume, $A$ halves, and $R = L/(\sigma A)$.
5. Everywhere current flows, at the rate $p = \mathbf{J}\cdot\mathbf{E} = J^2/\sigma$ per unit volume, so most strongly where the current crowds.
6. $\nabla\cdot\mathbf{J} + \partial\rho/\partial t = 0$. For steady currents, $\nabla\cdot\mathbf{J} = 0$: as much current leaves every region as enters it.
7. The normal current density is continuous (charge cannot pile up in a steady state). The normal field jumps by the ratio of conductivities, and a thin layer of surface charge at the junction, $\epsilon_0$ times the jump in $E_n$, produces that jump.